# Qwen3 (4B) Instruct — 양자화 인지 학습(QAT)

In [ ]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B를 2배 빠르게 실행
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 높은 정확도와 낮은 메모리 사용량을 위한 4비트 동적 양자화
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [신규] TTS 모델도 지원합니다!
] # 더 많은 모델: https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,   # 긴 문맥에 맞춰 원하는 값으로 설정하세요!
    load_in_4bit = False,    # 메모리 사용량을 줄이는 4비트 양자화
    load_in_8bit = False,    # [신규!] 정확도가 조금 더 높지만 메모리를 2배 사용합니다
    full_finetuning = False, # [신규!] 전체 매개변수 미세 조정도 지원합니다!
    # token = "YOUR_HF_TOKEN", # 접근 제한 모델용 HF 토큰
)

이제 LoRA 어댑터를 추가합니다. 이렇게 하면 소수의 매개변수만 업데이트하면 됩니다!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 0보다 큰 값을 선택하세요! 권장값: 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # 모든 값을 지원하지만 0에 최적화되어 있습니다
    bias = "none",    # 모든 값을 지원하지만 "none"에 최적화되어 있습니다
    # [신규] "unsloth"는 VRAM을 30% 적게 사용해 배치 크기를 2배로 늘릴 수 있습니다!
    qat_scheme = "int4",
    use_gradient_checkpointing = "unsloth", # 매우 긴 문맥에는 True 또는 "unsloth" 사용
    random_state = 3407,
    use_rslora = False,  # 순위 안정화 LoRA를 지원합니다
    loftq_config = None, # LoftQ도 지원합니다
)

QAT가 적용되었는지 확인해 보겠습니다!

In [ ]:
for module in model.modules():
    if "FakeQuantized" in module.__class__.__name__:
        print("QAT is applied!")
        break

<a name="Data"></a>
### 데이터 준비
대화형 미세 조정에는 `Qwen-3` 형식을 사용합니다. ShareGPT 형식의 [Maxime Labonne의 FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) 데이터셋을 사용합니다. Qwen-3는 여러 턴의 대화를 다음과 같이 표현합니다.

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
`get_chat_template` 함수로 알맞은 채팅 템플릿을 가져옵니다. `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` 등 다양한 형식을 지원합니다.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

이제 `standardize_data_formats`로 데이터셋을 미세 조정에 맞는 형식으로 변환합니다!

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

인덱스 100의 행이 어떻게 생겼는지 확인해 보겠습니다!

In [ ]:
dataset[100]

이제 대화에 `Qwen-3` 채팅 템플릿을 적용하고 결과를 `text`에 저장합니다.

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

채팅 템플릿이 적용된 결과를 확인해 보겠습니다!

In [ ]:
dataset[100]['text']

<a name="Train"></a>
### 모델 학습
이제 모델을 학습합니다. 빠르게 실행하기 위해 60스텝만 학습하지만, 전체 학습을 실행하려면 `num_train_epochs=1`로 설정하고 `max_steps=None`으로 스텝 제한을 해제하면 됩니다.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # 평가도 설정할 수 있습니다!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # 그래디언트 누적으로 더 큰 배치 크기를 구현하세요!
        warmup_steps = 5,
        # num_train_epochs = 1, # 전체 데이터를 한 번 학습하려면 설정하세요.
        max_steps = 30,
        learning_rate = 2e-4, # 장시간 학습할 때는 2e-5로 낮추세요
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # WandB 등을 사용하려면 설정하세요
    ),
)

Unsloth의 `train_on_completions` 메서드로 어시스턴트 출력만 학습하고 사용자 입력의 손실은 무시합니다. 이렇게 하면 미세 조정 정확도를 높일 수 있습니다! 이제 Unsloth가 토크나이저의 채팅 템플릿에서 지시문과 응답 부분을 자동으로 감지하므로 `instruction_part`와 `response_part`를 전달할 필요가 없습니다. 사용자 정의 채팅 템플릿을 사용한다면 여전히 명시적으로 전달할 수 있습니다.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

지시문 부분이 마스킹되었는지 확인하겠습니다! 인덱스 100의 행을 다시 출력합니다.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

이제 마스킹된 예제를 출력합니다. 답변만 남아 있어야 합니다.

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title 현재 메모리 사용 통계 표시
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

모델을 학습해 보겠습니다! 이전 학습을 재개하려면 `trainer.train(resume_from_checkpoint = True)`로 설정하세요.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title 최종 메모리 및 시간 통계 표시
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

학습이 끝났으므로 FakeQuantizedLinear 층을 표준 nn.Linear 층으로 되돌립니다. 이렇게 하면 가짜 양자화의 오버헤드를 제거하고 최종 변환이나 LoRA 어댑터 병합을 준비할 수 있습니다.

In [ ]:
from torchao.quantization import quantize_
from torchao.quantization.qat import QATConfig

quantize_(model, QATConfig(step = "convert"))

<a name="Inference"></a>
### 추론
Unsloth의 기본 추론 기능으로 모델을 실행해 보겠습니다! `Qwen-3` 팀이 권장하는 지시 응답 추론 설정은 `temperature = 0.7, top_p = 0.8, top_k = 20`입니다.

추론 과정을 포함하는 채팅 기반 추론에는 `temperature = 0.6, top_p = 0.95, top_k = 20`을 사용합니다.

In [ ]:
messages = [
    {"role" : "user", "content" : "Continue the sequence: 1, 1, 2, 3, 5, 8,"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # 생성 시 반드시 추가해야 합니다
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1000, # 더 긴 출력을 원하면 늘리세요!
    temperature = 0.7, top_p = 0.8, top_k = 20, # 비추론 모드용
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
### 미세 조정 모델 저장 및 불러오기
최종 모델을 LoRA 어댑터로 저장하려면 Hugging Face의 `push_to_hub`로 온라인에 저장하거나 `save_pretrained`로 로컬에 저장하세요.

**[참고]** 전체 모델이 아닌 LoRA 어댑터만 저장합니다. 16비트 또는 GGUF로 저장하려면 아래 내용을 참고하세요!

In [ ]:
model.save_pretrained("qwen_lora")  # 로컬에 저장
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # 온라인에 저장
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # 온라인에 저장

방금 저장한 LoRA 어댑터를 추론용으로 불러오려면 `False`를 `True`로 바꾸세요.

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora", # 학습에 사용한 모델
        max_seq_length = 2048,
        load_in_4bit = True,
    )

이제 TorchAO로 최종 모델을 저장하고 양자화할 수 있습니다. QAT 학습에 사용한 것과 동일한 설정을 적용합니다.

In [ ]:
model.save_pretrained_torchao(
    "model",
    tokenizer,
)

### TorchAO 내보내기 및 변환

사용자 정의 설정을 적용한 TorchAO 양자화 체크포인트로 내보내 vLLM이나 다른 추론 엔진에서 실행하는 기능도 지원합니다.

TorchAO 설정에 관한 자세한 내용은 Hugging Face Transformers 공식 문서를 참고하세요: https://huggingface.co/docs/transformers/main/quantization/torchao

In [ ]:
# TorchAO int4로 저장:
if False:
    from torchao.quantization import Int4WeightOnlyConfig
    model.save_pretrained_torchao("model", tokenizer, torchao_config = Int4WeightOnlyConfig())

if False: # HF Hub에 업로드
    from torchao.quantization import Int4WeightOnlyConfig
    model.save_pretrained_torchao(
        "HF_USERNAME/model", # hf를 본인의 사용자 이름으로 바꾸세요!
        tokenizer,
        torchao_config = Int4WeightOnlyConfig(),
        push_to_hub = True,
        token = "YOUR_HF_TOKEN", # 토큰 발급: https://huggingface.co/settings/tokens
    )

# TorchAO int8로 저장:
if False:
    from torchao.quantization import Int8DynamicActivationInt8WeightConfig
    model.save_pretrained_torchao("model", tokenizer, torchao_config = Int8DynamicActivationInt8WeightConfig(),)

if False: # HF Hub에 업로드
    from torchao.quantization import Int8DynamicActivationInt8WeightConfig
    model.save_pretrained_torchao(
        "HF_USERNAME/model", # hf를 본인의 사용자 이름으로 바꾸세요!
        tokenizer,
        torchao_config = Int8DynamicActivationInt8WeightConfig(),
        push_to_hub = True,
        token = "YOUR_HF_TOKEN", # 토큰 발급: https://huggingface.co/settings/tokens
    )

이제 모든 과정이 끝났습니다! Unsloth에 관한 질문은 [Discord](https://discord.gg/unsloth) 채널에서 나눌 수 있습니다. 버그를 발견했거나 최신 LLM 소식, 도움, 프로젝트 참여 기회를 찾는다면 자유롭게 참여하세요!

추가 참고 자료:

1. 로컬에서 Unsloth를 사용하려면 [설치 안내](https://unsloth.ai/docs/get-started/install)를 참고하세요. Windows, Docker, AMD 및 Intel GPU에서의 설치 방법을 설명합니다.
2. [강화 학습 안내 및 노트북](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide)로 강화 학습 방법을 익혀 보세요.
3. [텍스트 음성 변환(TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 및 [비전](https://unsloth.ai/docs/basics/vision-fine-tuning) 모델 지원에 관한 안내와 노트북을 참고하세요.
4. [LLM 튜토리얼 목록](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms)에서 모델별 전용 안내를 찾아보세요.
5. 추론에 도움이 필요하다면 [추론 및 배포 안내](https://unsloth.ai/docs/basics/inference-and-deployment)를 참고하세요. vLLM, llama.cpp, Ollama 등의 사용법을 설명합니다.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  도움이 필요하면 Discord에 참여하세요 + ⭐️ <i><a href="https://github.com/unslothai/unsloth">Github</a>에서 별을 눌러 주세요 </i> ⭐️

  이 노트북을 포함한 모든 Unsloth 노트북의 라이선스는 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)입니다
</div>